# Momentum: phase closure, energy, and a certificate

We solve $\ddot W+\gamma\dot W=-G W$ by evolving the full covariance in $(u,v,p,q)$ order. Initial velocity is zero. Predictor loss may oscillate even though mechanical energy decreases.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from linear_networks import QuadraticLoss, white_phase_covariance, integrate_momentum
from linear_networks.certificates import white_bootstrap, tightened_momentum_rate

loss = QuadraticLoss([[1.0]])
sigma_vu, sigma_qp = 25.0, 0.0
phase0 = white_phase_covariance(1, 1, sigma_vu, sigma_qp)
times = np.linspace(0.0, 3.0, 1001)
dampings = [5.0, 20.0, 60.0]
paths = {gamma: integrate_momentum(phase0, loss, times, gamma) for gamma in dampings}
for path in paths.values():
    assert np.max(np.diff(path.energies)) < 1e-8
print(f'Initial mechanical excess energy: {paths[20.0].energies[0]:.4f}')

## Distinguish predictor loss from mechanical energy

The energy is the predictor-loss gap plus $\tfrac12\mathrm{tr}(\Sigma^{(p,q)})$. Only the displayed curves are floored at $10^{-14}$ to make logarithmic plots readable; numerical checks use the unmodified arrays. Once the energy approaches the solver's absolute accuracy, computed tails can become nonmonotone or slightly negative. This is a numerical resolution limit, not physical energy growth; repeat with tighter tolerances to assess it.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.3), constrained_layout=True)
for gamma, path in paths.items():
    axes[0].semilogy(times, np.maximum(path.loss_gaps, 1e-14), label=f'gamma={gamma:g}')
    axes[1].semilogy(times, np.maximum(path.energies, 1e-14), label=f'gamma={gamma:g}')
axes[0].set(xlabel='time', ylabel='predictor-loss gap', ylim=(1e-14, 1))
axes[1].set(xlabel='time', ylabel='mechanical excess energy', ylim=(1e-14, 1))
for ax in axes:
    ax.legend()
plt.show()

## Test an initialization-only certificate

For white initial position covariance and zero position-velocity cross covariance, a **strictly negative** bootstrap gap certifies the tube. A nonnegative gap is inconclusive; it does not mean that training fails. After certifying a tube, the tightened bound is $\mathcal E_0(1-q)^{-1}e^{-\lambda t}$. No rate is fitted to the trajectory.

In [ ]:
budget = 1.0 / 3.0
energy0 = loss.gap(np.zeros((1, 1))) + 0.5 * np.trace(phase0[2:, 2:])
for gamma, path in paths.items():
    gap, conservative_rate = white_bootstrap(
        budget, sigma_vu, gamma, kappa=loss.kappa,
        smoothness=loss.smoothness, energy=energy0,
    )
    print(f'gamma={gamma:g}: bootstrap gap={float(gap):.4f}')
    if gap < 0:
        a, b = sigma_vu**2 * np.exp(-2*budget), sigma_vu**2 * np.exp(2*budget)
        rate, q = tightened_momentum_rate(a, b, gamma, kappa=loss.kappa,
                                          smoothness=loss.smoothness, energy=energy0)
        envelope = energy0 / (1-q) * np.exp(-rate * times)
        assert np.max(path.energies - envelope) < 1e-8
        print(f'  certified tightened rate={rate:.4f}, q={q:.4f}')

Try a smaller initial position scale or nonzero initial velocity and recompute the energy used in the certificate. The numerical solver supports zero damping, but the positive-damping convergence certificate does not apply there.